# First Model: CatBoost with Feature Generator (Single-Window Run)

Pipeline:
1. Load train/test data (full 100-candle sessions for train)
2. Parse headlines → template_id + template sentiment
3. Build one training row per session: input 0..49, target 49->99
4. Train CatBoost with GroupKFold (sessions never split across folds)
5. Predict on public + private test and submit



In [102]:
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import GroupKFold
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")

from feature_generator import generate_features

DATA_DIR = Path("hrt-eth-zurich-datathon-2026/data")
N_TEMPLATES = 50
RECENCY_DECAY = 0.05
HORIZON = 50
INPUT_END_BAR = 49
TARGET_BAR = 99
SENTIMENT_K = 5
SENTIMENT_PRIOR_N = 30.0



## 1. Load raw data

In [103]:
# Train — full 100-candle sessions (seen + unseen)
bars_seen = pd.read_parquet(DATA_DIR / "bars_seen_train.parquet")
bars_unseen = pd.read_parquet(DATA_DIR / "bars_unseen_train.parquet")
bars_full = pd.concat([bars_seen, bars_unseen]).sort_values(["session", "bar_ix"]).reset_index(drop=True)

# Test — seen only (50 candles)
bars_pub = pd.read_parquet(DATA_DIR / "bars_seen_public_test.parquet")
bars_priv = pd.read_parquet(DATA_DIR / "bars_seen_private_test.parquet")

FEATURE_PARQUET_BY_SPLIT = {
    "train": [Path("headline_features_train.parquet"), Path("headline_features.parquet")],
    "public": [Path("headline_features_public.parquet")],
    "private": [Path("headline_features_private.parquet")],
}
PARSER_JSON_BY_SPLIT = {
    "train": Path("title_template_slot_extractions_parser.json"),
    "public": Path("title_template_slot_extractions_parser_public.json"),
    "private": Path("title_template_slot_extractions_parser_private.json"),
}
EXPECTED_SESSION_RANGES = {
    "train": (0, 999),
    "public": (1000, 10999),
    "private": (11000, 20999),
}


def load_headline_features(split: str):
    required = ["session", "bar_ix", "template_index", "dollar", "percentage"]
    source = None
    df = None

    for path in FEATURE_PARQUET_BY_SPLIT[split]:
        if path.exists():
            df = pd.read_parquet(path)
            source = path.name
            break

    if df is None:
        parser_path = PARSER_JSON_BY_SPLIT[split]
        if not parser_path.exists():
            raise FileNotFoundError(
                f"No preprocessed parquet found for split={split} and parser JSON missing: {parser_path}"
            )
        records = json.loads(parser_path.read_text(encoding="utf-8"))
        df = pd.DataFrame(records)
        source = parser_path.name

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{split}: missing required columns {missing} in {source}")

    session_min, session_max = EXPECTED_SESSION_RANGES[split]
    smin = int(pd.to_numeric(df["session"], errors="coerce").min())
    smax = int(pd.to_numeric(df["session"], errors="coerce").max())
    if smin < session_min or smax > session_max:
        raise ValueError(
            f"{split}: expected sessions in [{session_min}, {session_max}], got [{smin}, {smax}] from {source}"
        )

    return df, source


# Split-specific parsed headline features (template_index, dollar, percentage)
hf_train, hf_train_src = load_headline_features("train")
hf_pub, hf_pub_src = load_headline_features("public")
hf_priv, hf_priv_src = load_headline_features("private")

print(f"Train:        {bars_full.session.nunique():>6} sessions x {bars_full.groupby('session').size().iloc[0]} bars")
print(f"Public test:  {bars_pub.session.nunique():>6} sessions x {bars_pub.groupby('session').size().iloc[0]} bars")
print(f"Private test: {bars_priv.session.nunique():>6} sessions x {bars_priv.groupby('session').size().iloc[0]} bars")
print(f"Headline sources: train={hf_train_src}, public={hf_pub_src}, private={hf_priv_src}")

Train:          1000 sessions x 100 bars
Public test:   10000 sessions x 50 bars
Private test:  10000 sessions x 50 bars
Headline sources: train=headline_features_train.parquet, public=headline_features_public.parquet, private=headline_features_private.parquet


## 2. Prepare headlines

In [104]:
def build_template_sentiment_map(
    headline_dfs,
    bars_dfs,
    k: int = SENTIMENT_K,
    prior_n: float = SENTIMENT_PRIOR_N,
    max_headline_bar: int = INPUT_END_BAR,
):
    """Template sentiment = shrunk mean forward K-bar return across provided splits."""
    hl = pd.concat(
        [df[["session", "bar_ix", "template_index"]] for df in headline_dfs],
        ignore_index=True,
    )
    hl = hl.dropna(subset=["session", "bar_ix", "template_index"]).copy()
    hl["session"] = pd.to_numeric(hl["session"], errors="coerce").astype(int)
    hl["bar_ix"] = pd.to_numeric(hl["bar_ix"], errors="coerce").astype(int)
    hl["template_index"] = pd.to_numeric(hl["template_index"], errors="coerce").astype(int)
    hl = hl[hl["bar_ix"].between(0, max_headline_bar)].copy()

    bars = pd.concat(
        [df[["session", "bar_ix", "close"]] for df in bars_dfs],
        ignore_index=True,
    ).drop_duplicates(["session", "bar_ix"])

    close = bars.set_index(["session", "bar_ix"])["close"].sort_index()
    max_bar = bars.groupby("session")["bar_ix"].max()

    end_bars = np.minimum(
        hl["bar_ix"].to_numpy() + int(k),
        hl["session"].map(max_bar).to_numpy(),
    )

    idx0 = list(zip(hl["session"].to_numpy(), hl["bar_ix"].to_numpy()))
    idx1 = list(zip(hl["session"].to_numpy(), end_bars))

    hl["c0"] = close.reindex(idx0).to_numpy()
    hl["c1"] = close.reindex(idx1).to_numpy()
    hl["fwd_ret_k"] = hl["c1"] / hl["c0"] - 1.0
    hl = hl[np.isfinite(hl["fwd_ret_k"])].copy()

    global_mean = float(hl["fwd_ret_k"].mean()) if len(hl) else 0.0

    agg = (
        hl.groupby("template_index")["fwd_ret_k"]
        .agg(["count", "mean"])
        .reset_index()
    )
    agg["sentiment"] = (
        agg["count"] * agg["mean"] + float(prior_n) * global_mean
    ) / (agg["count"] + float(prior_n))

    sentiment_map = agg.set_index("template_index")["sentiment"].to_dict()
    return sentiment_map, global_mean, agg.sort_values("count", ascending=False)


def prepare_headlines(hf_df, template_sentiment_map, global_sentiment):
    """Convert parsed headlines for feature generation.

    Dollar/percentage amplitudes are intentionally removed for this comparison run.
    """
    df = hf_df[["session", "bar_ix", "template_index"]].copy()
    df = df.rename(columns={"bar_ix": "candle_idx", "template_index": "template_id"})

    df["session"] = pd.to_numeric(df["session"], errors="coerce").astype(int)
    df["candle_idx"] = pd.to_numeric(df["candle_idx"], errors="coerce").astype(int)
    df["template_id"] = pd.to_numeric(df["template_id"], errors="coerce").fillna(-1).astype(int)

    # Remove dollar/percentage signal completely.
    df["amplitude"] = 1.0

    df["sentiment"] = (
        df["template_id"].map(template_sentiment_map)
        .fillna(float(global_sentiment))
        .astype(float)
    )

    df = df[df["candle_idx"].between(0, INPUT_END_BAR)].copy()
    return df[["session", "candle_idx", "template_id", "amplitude", "sentiment"]]


TEMPLATE_SENTIMENT_MAP, GLOBAL_TEMPLATE_SENTIMENT, template_sent_stats = build_template_sentiment_map(
    headline_dfs=[hf_train, hf_pub, hf_priv],
    bars_dfs=[bars_full, bars_pub, bars_priv],
    k=SENTIMENT_K,
    prior_n=SENTIMENT_PRIOR_N,
)

hl_train = prepare_headlines(hf_train, TEMPLATE_SENTIMENT_MAP, GLOBAL_TEMPLATE_SENTIMENT)
hl_pub = prepare_headlines(hf_pub, TEMPLATE_SENTIMENT_MAP, GLOBAL_TEMPLATE_SENTIMENT)
hl_priv = prepare_headlines(hf_priv, TEMPLATE_SENTIMENT_MAP, GLOBAL_TEMPLATE_SENTIMENT)

print(
    f"Template sentiment map: {len(TEMPLATE_SENTIMENT_MAP)} templates, "
    f"global={GLOBAL_TEMPLATE_SENTIMENT:.6f}, K={SENTIMENT_K}, prior_n={SENTIMENT_PRIOR_N}"
)
print(
    f"Headlines prepared: train={len(hl_train)}, public={len(hl_pub)}, private={len(hl_priv)}"
)



Template sentiment map: 50 templates, global=0.000440, K=5, prior_n=30.0
Headlines prepared: train=9740, public=99308, private=99148


## 3. Build single-window training data (1 row per session)

For each of the 1000 sessions, use candles 0..49 as model input and predict the forward return from candle 49 to candle 99.

This gives exactly **1000 training rows** (no x5 history-length expansion).



In [105]:
def build_feature_matrix(bars_df, headlines_df, label=""):
    """Generate features for every session using first 50 candles only."""
    sessions = sorted(bars_df["session"].unique())
    bars_grouped = {s: g.sort_values("bar_ix").reset_index(drop=True) for s, g in bars_df.groupby("session")}
    hl_grouped = dict(list(headlines_df.groupby("session")))
    empty_hl = pd.DataFrame(columns=["candle_idx", "template_id", "amplitude", "sentiment"])

    rows = {}
    t0 = time.time()
    for i, s in enumerate(sessions):
        candles = bars_grouped[s].iloc[: INPUT_END_BAR + 1].reset_index(drop=True)
        hl = hl_grouped.get(s, empty_hl)
        feats = generate_features(candles, hl, n_templates=N_TEMPLATES, recency_decay=RECENCY_DECAY)
        rows[s] = feats
        if (i + 1) % 2000 == 0:
            print(f"  {label} {i+1}/{len(sessions)} ({(i+1)/(time.time()-t0):.0f} sess/s)")

    X = pd.DataFrame(rows).T
    X.index.name = "session"
    print(f"  {label} done: {X.shape[0]}x{X.shape[1]} in {time.time()-t0:.1f}s")
    return X


def build_single_h50_training(bars_full_df, headlines_df):
    """Build one row per session: input bars 0..49, target return 49->99."""
    sessions = sorted(bars_full_df["session"].unique())
    bars_grouped = {s: g.sort_values("bar_ix").reset_index(drop=True)
                    for s, g in bars_full_df.groupby("session")}
    hl_grouped = dict(list(headlines_df.groupby("session")))
    empty_hl = pd.DataFrame(columns=["candle_idx", "template_id", "amplitude", "sentiment"])

    all_rows = []
    t0 = time.time()
    for i, s in enumerate(sessions):
        candles = bars_grouped[s]
        if len(candles) <= TARGET_BAR:
            continue

        candles_in = candles.iloc[: INPUT_END_BAR + 1].reset_index(drop=True)
        hl = hl_grouped.get(s, empty_hl)
        hl = hl[hl["candle_idx"].between(0, INPUT_END_BAR)].copy()

        feats = generate_features(candles_in, hl, n_templates=N_TEMPLATES, recency_decay=RECENCY_DECAY)
        c49 = float(candles.iloc[INPUT_END_BAR]["close"])
        c99 = float(candles.iloc[TARGET_BAR]["close"])
        feats["target"] = c99 / c49 - 1.0
        feats["session_id"] = int(s)
        all_rows.append(feats)

        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(sessions)} ({(i+1)/(time.time()-t0):.0f} sess/s)")

    df = pd.DataFrame(all_rows)
    print(f"  Done: {df.shape[0]} rows x {df.shape[1]} cols in {time.time()-t0:.1f}s")
    return df



In [106]:
print("Building single-window training data (1 row per session, 0..49 -> 50..99)...")
train_df = build_single_h50_training(bars_full, hl_train)

# Separate features, target, groups
meta_cols = ["target", "session_id"]
feature_cols = [c for c in train_df.columns if c not in meta_cols]
X_train = train_df[feature_cols]
y_train = train_df["target"]
groups = train_df["session_id"]

print()
print(f"Training set: {X_train.shape[0]} rows x {X_train.shape[1]} features")
print("Target stats:")
print(y_train.describe())



Building single-window training data (1 row per session, 0..49 -> 50..99)...
  200/1000 (160 sess/s)
  400/1000 (159 sess/s)
  600/1000 (160 sess/s)
  800/1000 (160 sess/s)
  1000/1000 (160 sess/s)
  Done: 1000 rows x 263 cols in 6.3s

Training set: 1000 rows x 261 features
Target stats:
count    1000.000000
mean        0.003531
std         0.020434
min        -0.072020
25%        -0.009112
50%         0.003706
75%         0.016982
max         0.080883
Name: target, dtype: float64


## 4. RFE feature selection with CatBoost

Iteratively train CatBoost (GroupKFold CV), drop the bottom 25% of features by average importance, and repeat. Track OOF Sharpe at each feature count — pick the subset size that maximizes it.

In [107]:
def sharpe(positions, returns):
    """Competition Sharpe ratio."""
    pnl = positions * returns
    std = pnl.std()
    if std < 1e-12:
        return 0.0
    return float(pnl.mean() / std * 16)


N_FOLDS = 5
N_ROUNDS = 1800
EARLY_STOP = 150


def run_cv(X, y, groups, params, n_rounds=N_ROUNDS, early_stop=EARLY_STOP, n_folds=N_FOLDS):
    """GroupKFold CV. Returns (oof_sharpe, fold_mean, fold_std, mean_importance, models, oof_preds)."""
    gkf = GroupKFold(n_splits=n_folds)
    oof = np.full(len(X), np.nan)
    fold_sharpes_local = []
    imps = []
    models_local = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        train_pool = Pool(X_tr, label=y_tr)
        val_pool = Pool(X_va, label=y_va)

        model = CatBoostRegressor(
            iterations=n_rounds,
            early_stopping_rounds=early_stop,
            eval_metric="RMSE",
            use_best_model=True,
            **params,
        )
        model.fit(train_pool, eval_set=val_pool, verbose=False)

        pred_va = model.predict(X_va)
        oof[va_idx] = pred_va

        fs = sharpe(pred_va, y_va.values)
        fold_sharpes_local.append(fs)
        imps.append(model.get_feature_importance())
        models_local.append(model)

    oof_s = sharpe(oof, y.values)
    mean_imp = pd.Series(np.mean(imps, axis=0), index=X.columns)
    return (
        oof_s,
        float(np.mean(fold_sharpes_local)),
        float(np.std(fold_sharpes_local)),
        mean_imp,
        models_local,
        oof,
    )


# Slightly more conservative base params to reduce overfitting
cat_params = {
    "loss_function": "RMSE",
    "learning_rate": 0.04,
    "depth": 6,
    "min_data_in_leaf": 25,
    "l2_leaf_reg": 8.0,
    "bootstrap_type": "Bernoulli",
    "subsample": 0.85,
    "rsm": 0.85,
    "random_strength": 0.5,
    "random_seed": 42,
    "verbose": False,
    "allow_writing_files": False,
}

# RFE loop: drop bottom DROP_FRAC each round until MIN_FEATURES
DROP_FRAC = 0.20
MIN_FEATURES = 40

current_features = list(X_train.columns)
rfe_history = []
t0 = time.time()
while True:
    X_cur = X_train[current_features]
    oof_s, mean_s, std_s, mean_imp, _, _ = run_cv(
        X_cur, y_train, groups, cat_params
    )
    rfe_history.append({
        "n_features": len(current_features),
        "oof_sharpe": oof_s,
        "fold_mean": mean_s,
        "fold_std": std_s,
        "features": current_features.copy(),
    })
    print(f"  RFE n_feats={len(current_features):3d}: OOF Sharpe={oof_s:.3f}, fold={mean_s:.3f}±{std_s:.3f}")

    if len(current_features) <= MIN_FEATURES:
        break

    n_keep = max(int(round(len(current_features) * (1 - DROP_FRAC))), MIN_FEATURES)
    keep = mean_imp.sort_values(ascending=False).head(n_keep).index.tolist()
    current_features = keep

print()
print(f"RFE done: {len(rfe_history)} rounds in {time.time()-t0:.1f}s")



  RFE n_feats=261: OOF Sharpe=2.823, fold=2.941±0.719
  RFE n_feats=196: OOF Sharpe=2.671, fold=2.771±0.602
  RFE n_feats=147: OOF Sharpe=2.883, fold=3.073±0.672
  RFE n_feats=110: OOF Sharpe=2.937, fold=3.061±0.551
  RFE n_feats= 82: OOF Sharpe=3.086, fold=3.160±0.549
  RFE n_feats= 62: OOF Sharpe=3.122, fold=3.261±0.434
  RFE n_feats= 46: OOF Sharpe=3.255, fold=3.455±0.358
  RFE n_feats= 34: OOF Sharpe=3.515, fold=3.592±0.201
  RFE n_feats= 26: OOF Sharpe=3.202, fold=3.389±0.257
  RFE n_feats= 20: OOF Sharpe=3.636, fold=3.781±0.313

RFE done: 10 rounds in 23.0s


## 5. Pick best feature subset from RFE history

In [108]:
rfe_df = pd.DataFrame([{k: v for k, v in r.items() if k != "features"} for r in rfe_history])
print("RFE history:")
print(rfe_df.to_string(index=False))

# Conservative selection: smallest feature set within tolerance of best OOF Sharpe
RFE_OOF_TOL = 0.03
best_oof = max(r["oof_sharpe"] for r in rfe_history)
cands = [r for r in rfe_history if r["oof_sharpe"] >= best_oof - RFE_OOF_TOL]
best_round = min(cands, key=lambda r: r["n_features"])
best_features = best_round["features"]

print()
print(
    f"Best absolute RFE OOF Sharpe: {best_oof:.3f} | "
    f"chosen tolerance={RFE_OOF_TOL:.3f}"
)
print(
    f"Chosen RFE round: n_feats={best_round['n_features']}, "
    f"OOF Sharpe={best_round['oof_sharpe']:.3f}, "
    f"fold={best_round['fold_mean']:.3f}±{best_round['fold_std']:.3f}"
)
print()
print(f"Selected {len(best_features)} features.")



RFE history:
 n_features  oof_sharpe  fold_mean  fold_std
        261    2.822556   2.940835  0.719391
        196    2.671203   2.770833  0.602226
        147    2.882861   3.073017  0.671731
        110    2.937142   3.061250  0.550869
         82    3.086330   3.159951  0.549178
         62    3.122077   3.260928  0.434300
         46    3.254899   3.455373  0.358291
         34    3.514649   3.592168  0.201172
         26    3.202104   3.389496  0.257467
         20    3.635940   3.780598  0.313462

Best RFE round: n_feats=20, OOF Sharpe=3.636, fold=3.781±0.313

Selected 20 features.


## 5b. Hyperparameter search + final retrain

Random-search CatBoost's core knobs (`learning_rate`, `depth`, `l2_leaf_reg`, `min_data_in_leaf`) using OOF Sharpe as the objective on the RFE-selected features, then retrain CV with the winner to produce `models_active` for inference.

In [109]:
from itertools import product

X_train_best = X_train[best_features].copy()

# Conservative hyperparameter ranges (avoid very high-capacity corners)
search_space = {
    "learning_rate": [0.02, 0.03, 0.05],
    "depth": [4, 5, 6],
    "l2_leaf_reg": [6.0, 10.0, 15.0],
    "min_data_in_leaf": [20, 35, 50],
}

N_TRIALS = 18
rng = np.random.default_rng(2026)
all_combos = [dict(zip(search_space.keys(), v)) for v in product(*search_space.values())]
rng.shuffle(all_combos)
trials = all_combos[:N_TRIALS]

results = []
t0 = time.time()
for i, p in enumerate(trials):
    params = {**cat_params, **p}
    oof_s, mean_s, std_s, _, _, _ = run_cv(
        X_train_best, y_train, groups, params
    )
    robust_s = mean_s - 0.35 * std_s
    results.append({
        **p,
        "oof_sharpe": oof_s,
        "fold_mean": mean_s,
        "fold_std": std_s,
        "robust_score": robust_s,
    })
    print(
        f"  Trial {i+1:2d}/{len(trials)}: lr={p['learning_rate']}, depth={p['depth']}, "
        f"l2={p['l2_leaf_reg']}, mdl={p['min_data_in_leaf']} -> "
        f"OOF={oof_s:.3f}, robust={robust_s:.3f}"
    )

results_df = pd.DataFrame(results).sort_values(
    ["robust_score", "oof_sharpe"], ascending=False
).reset_index(drop=True)
print()
print(f"Hyperparam search: {time.time()-t0:.1f}s")
print()
print("Top 5 configs (by robust_score):")
print(results_df.head().to_string(index=False))

best_hp = {k: results_df.iloc[0][k] for k in search_space.keys()}
best_hp["depth"] = int(best_hp["depth"])
best_hp["min_data_in_leaf"] = int(best_hp["min_data_in_leaf"])
best_hp["learning_rate"] = float(best_hp["learning_rate"])
best_hp["l2_leaf_reg"] = float(best_hp["l2_leaf_reg"])
print()
print(f"Best hyperparams: {best_hp}")

# Final CV retrain with best features + best hyperparams
final_params = {**cat_params, **best_hp}
oof_final, mean_final, std_final, _, models_active, oof_preds_top = run_cv(
    X_train_best, y_train, groups, final_params
)

print()
print(f"Final OOF Sharpe: {oof_final:.3f}")
print(f"Final fold Sharpe: {mean_final:.3f} +/- {std_final:.3f}")

feature_cols_active = best_features
print()
print(f"Inference model: {len(feature_cols_active)} features, {len(models_active)} folds")



  Trial  1/15: lr=0.05, depth=5, l2=1.0, mdl=20 -> Sharpe=3.624
  Trial  2/15: lr=0.05, depth=8, l2=3.0, mdl=20 -> Sharpe=3.173
  Trial  3/15: lr=0.05, depth=5, l2=6.0, mdl=20 -> Sharpe=3.633
  Trial  4/15: lr=0.08, depth=6, l2=6.0, mdl=5 -> Sharpe=3.441
  Trial  5/15: lr=0.08, depth=8, l2=3.0, mdl=5 -> Sharpe=3.140
  Trial  6/15: lr=0.1, depth=6, l2=6.0, mdl=5 -> Sharpe=3.262
  Trial  7/15: lr=0.03, depth=7, l2=1.0, mdl=5 -> Sharpe=3.725
  Trial  8/15: lr=0.08, depth=7, l2=6.0, mdl=5 -> Sharpe=3.370
  Trial  9/15: lr=0.1, depth=8, l2=1.0, mdl=10 -> Sharpe=3.294
  Trial 10/15: lr=0.08, depth=5, l2=3.0, mdl=20 -> Sharpe=3.446
  Trial 11/15: lr=0.08, depth=5, l2=3.0, mdl=10 -> Sharpe=3.446
  Trial 12/15: lr=0.05, depth=7, l2=6.0, mdl=5 -> Sharpe=3.533
  Trial 13/15: lr=0.1, depth=7, l2=1.0, mdl=10 -> Sharpe=3.364
  Trial 14/15: lr=0.08, depth=8, l2=1.0, mdl=10 -> Sharpe=3.193
  Trial 15/15: lr=0.1, depth=6, l2=3.0, mdl=10 -> Sharpe=3.391

Hyperparam search: 15.9s

Top 5 configs:
 learnin

## 5c. Optimize output transformation on OOF predictions

Grid-search the position-shaping knobs (`threshold_q`, `blend_alpha`, `short_floor`) against OOF Sharpe on all training rows.
Since OOF preds come from the final CV retrain, they reflect held-out predictions per session — so we can tune shaping without re-running training.



In [110]:
def pick_vol_proxy(X: pd.DataFrame):
    """Pick a volatility proxy column for position scaling."""
    candidates = [
        "return_std_w50", "realized_vol_w50", "atr_w50",
        "return_std_w20", "realized_vol_w20",
    ]
    for col in candidates:
        if col in X.columns:
            v = pd.to_numeric(X[col], errors="coerce").to_numpy(dtype=float)
            finite = np.isfinite(v)
            fill = float(np.nanmedian(v[finite])) if finite.any() else 1.0
            v = np.nan_to_num(v, nan=fill, posinf=fill, neginf=fill)
            v = np.maximum(np.abs(v), 1e-6)
            return v, col

    std_cols = [c for c in X.columns if c.startswith("return_std_w")]
    if std_cols:
        v = X[std_cols].mean(axis=1).to_numpy(dtype=float)
        finite = np.isfinite(v)
        fill = float(np.nanmedian(v[finite])) if finite.any() else 1.0
        v = np.nan_to_num(v, nan=fill, posinf=fill, neginf=fill)
        v = np.maximum(np.abs(v), 1e-6)
        return v, "mean(return_std_w*)"

    return np.ones(len(X), dtype=float), "ones"


def shape_positions_thresholded_inv_vol(
    pred: np.ndarray, vol: np.ndarray,
    threshold_q: float = 0.35, blend_alpha: float = 0.5, short_floor: float = 0.3,
):
    """1) threshold small |pred|  2) inverse-vol scale  3) normalize mean(|pos|)=1
    4) blend with constant long  5) floor shorts."""
    pred = np.asarray(pred, dtype=float)
    vol = np.asarray(vol, dtype=float)
    abs_pred = np.abs(pred)
    cutoff = float(np.quantile(abs_pred, threshold_q)) if threshold_q > 0 else 0.0
    pos = pred.copy()
    pos[abs_pred < cutoff] = 0.0
    pos = pos / np.maximum(vol, 1e-6)
    m = float(np.mean(np.abs(pos)))
    if m > 1e-12:
        pos = pos / m
    pos = blend_alpha * pos + (1.0 - blend_alpha) * 1.0
    pos = np.maximum(pos, short_floor)
    return pos, cutoff


# Tighter shaping grid to reduce overfitting risk
oof_train = oof_preds_top
y_train_eval = y_train.values
vol_train_full, vol_col = pick_vol_proxy(X_train)

shape_grid = {
    "threshold_q": [0.0, 0.15, 0.30, 0.50],
    "blend_alpha": [0.50, 0.75, 1.0],
    "short_floor": [0.0, 0.2, 0.3],
}

shape_results = []
for tq in shape_grid["threshold_q"]:
    for ba in shape_grid["blend_alpha"]:
        for sf in shape_grid["short_floor"]:
            pos, _ = shape_positions_thresholded_inv_vol(
                oof_train, vol_train_full, threshold_q=tq, blend_alpha=ba, short_floor=sf
            )
            shape_results.append({
                "threshold_q": tq,
                "blend_alpha": ba,
                "short_floor": sf,
                "sharpe": sharpe(pos, y_train_eval),
            })

shape_df = pd.DataFrame(shape_results).sort_values("sharpe", ascending=False).reset_index(drop=True)
print(f"Vol proxy: {vol_col}  (OOF rows: {len(oof_train)})")
print(f"Baseline (raw OOF preds as positions): Sharpe={sharpe(oof_train, y_train_eval):.3f}")
print()
print("Top 10 shaping configs on OOF:")
print(shape_df.head(10).to_string(index=False))

best_shape = shape_df.iloc[0]
SHAPE_THRESHOLD_Q = float(best_shape["threshold_q"])
SHAPE_BLEND_ALPHA = float(best_shape["blend_alpha"])
SHAPE_SHORT_FLOOR = float(best_shape["short_floor"])
print()
print(
    f"Best shaping: threshold_q={SHAPE_THRESHOLD_Q}, "
    f"blend_alpha={SHAPE_BLEND_ALPHA}, short_floor={SHAPE_SHORT_FLOOR}"
)
print(f"Best OOF Sharpe: {best_shape['sharpe']:.3f}")



Vol proxy: return_std_w50  (OOF rows: 1000)
Baseline (raw OOF preds as positions): Sharpe=3.725

Top 10 shaping configs on OOF:
 threshold_q  blend_alpha  short_floor   sharpe
        0.00         0.75         -1.0 3.781571
        0.00         1.00         -1.0 3.754179
        0.00         0.75         -0.3 3.746622
        0.00         1.00         -0.3 3.732288
        0.20         0.75         -1.0 3.727650
        0.00         0.75          0.0 3.712873
        0.50         0.75         -1.0 3.711488
        0.35         0.75         -1.0 3.710908
        0.50         0.50         -1.0 3.695708
        0.20         0.75         -0.3 3.692102

Best shaping: threshold_q=0.0, blend_alpha=0.75, short_floor=-1.0
Best OOF Sharpe: 3.782


## 6. Generate test features & predict (public + private, h=50)

In [111]:
print(f"Inference model: {len(models_active)} folds, {len(feature_cols_active)} features")
print(f"Shaping: threshold_q={SHAPE_THRESHOLD_Q}, "
      f"blend_alpha={SHAPE_BLEND_ALPHA}, short_floor={SHAPE_SHORT_FLOOR}")

print("\nBuilding PUBLIC TEST features...")
X_pub = build_feature_matrix(bars_pub, hl_pub, label="Public")
X_pub = X_pub.reindex(columns=feature_cols_active, fill_value=np.nan)
preds_pub_raw = np.column_stack([m.predict(X_pub) for m in models_active]).mean(axis=1)
vol_pub, vol_col_pub = pick_vol_proxy(X_pub)
preds_pub, cut_pub = shape_positions_thresholded_inv_vol(
    preds_pub_raw, vol_pub,
    threshold_q=SHAPE_THRESHOLD_Q,
    blend_alpha=SHAPE_BLEND_ALPHA,
    short_floor=SHAPE_SHORT_FLOOR,
)
print(
    f"Public raw std={np.std(preds_pub_raw):.6f}, shaped std={np.std(preds_pub):.6f}, "
    f"vol_col={vol_col_pub}, cutoff={cut_pub:.6g}, frac positive={np.mean(preds_pub > 0):.1%}"
)

print("\nBuilding PRIVATE TEST features...")
X_priv = build_feature_matrix(bars_priv, hl_priv, label="Private")
X_priv = X_priv.reindex(columns=feature_cols_active, fill_value=np.nan)
preds_priv_raw = np.column_stack([m.predict(X_priv) for m in models_active]).mean(axis=1)
vol_priv, _ = pick_vol_proxy(X_priv)
preds_priv, cut_priv = shape_positions_thresholded_inv_vol(
    preds_priv_raw, vol_priv,
    threshold_q=SHAPE_THRESHOLD_Q,
    blend_alpha=SHAPE_BLEND_ALPHA,
    short_floor=SHAPE_SHORT_FLOOR,
)
print(
    f"Private raw std={np.std(preds_priv_raw):.6f}, shaped std={np.std(preds_priv):.6f}, "
    f"cutoff={cut_priv:.6g}, frac positive={np.mean(preds_priv > 0):.1%}"
)


Inference model: 5 folds, 20 features
Shaping: threshold_q=0.0, blend_alpha=0.75, short_floor=-1.0

Building PUBLIC TEST features...
  Public 2000/10000 (173 sess/s)
  Public 4000/10000 (171 sess/s)
  Public 6000/10000 (173 sess/s)
  Public 8000/10000 (173 sess/s)
  Public 10000/10000 (174 sess/s)
  Public done: 10000x261 in 57.6s
Public raw std=0.003465, shaped std=0.672269, vol_col=ones, cutoff=0, frac positive=91.9%

Building PRIVATE TEST features...
  Private 2000/10000 (163 sess/s)
  Private 4000/10000 (167 sess/s)
  Private 6000/10000 (170 sess/s)
  Private 8000/10000 (172 sess/s)
  Private 10000/10000 (173 sess/s)
  Private done: 10000x261 in 58.0s
Private raw std=0.003461, shaped std=0.671671, cutoff=0, frac positive=92.0%


## 7. Create combined submission (public + private)

In [112]:
sub_pub = pd.DataFrame({"session": X_pub.index, "target_position": preds_pub})
sub_priv = pd.DataFrame({"session": X_priv.index, "target_position": preds_priv})
submission = pd.concat([sub_pub, sub_priv], ignore_index=True).sort_values("session").reset_index(drop=True)

# Sanity checks
assert len(submission) == 20000, f"Expected 20000, got {len(submission)}"
assert submission["session"].is_unique, "Duplicate sessions"
assert submission["target_position"].notna().all(), "NaN in positions"
assert submission["session"].min() == 1000 and submission["session"].max() == 20999

submission.to_csv("submission_lgbm_featuregen.csv", index=False)
print(f"Submission saved: {len(submission)} sessions (public 1000-10999 + private 11000-20999)")
print(f"\n{submission.describe()}")

Submission saved: 20000 sessions (public 1000-10999 + private 11000-20999)

            session  target_position
count  20000.000000     20000.000000
mean   10999.500000         0.901815
std     5773.647028         0.671988
min     1000.000000        -1.000000
25%     5999.750000         0.435264
50%    10999.500000         0.861497
75%    15999.250000         1.328746
max    20999.000000         4.272678
